# Transformación y tratamiento de variables

## Transformación de variables
En el análisis de datos, la transformación de variables es una técnica utilizada para modificar la escala o distribución de los datos con el fin de mejorar la interpretación, visualización o modelado.


### Importacion de librerías

In [38]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler

### Cargue de datos

In [18]:
df_raw = pd.read_csv('../../data/raw/DATOS_PROCESADOS.csv')
df_raw.head()

,FECHA_HORA_REPORTE,aerodromo,fecha_zulu,viento,visibilidad,nubosidad,temperatura/rocio,presion,fenomenos,direccion_viento,velocidad_viento
0,2006-07-20 00:00:00,SKBQ,200000Z,03009KT,9999,"FEW015CB, SCT080",28/24,A2983,"TS, CB, TS",30.0,9.0
1,2006-07-20 01:00:00,SKBQ,200100Z,25008KT,9999,"FEW015CB, SCT080",27/23,A2985,TS,250.0,8.0
2,2006-07-20 05:00:00,SKBO,200500Z,VRB02KT,9999,"FEW017, SCT200",09/09,A3036,NaN,NaN,2.0
3,2006-07-20 06:00:00,SKBQ,200600Z,29006KT,9999,"FEW015, SCT080",26/23,A2985,NaN,290.0,6.0
4,2006-07-20 07:00:00,SKBQ,200700Z,32006KT,9999,SCT012,26/24,A2984,NaN,320.0,6.0


### Conversion de variables

In [99]:
def parse_viento_flexible(cadena):
    if pd.isna(cadena) or not isinstance(cadena, str):
        return pd.Series([None, None, None])
    
    patron = r'^(\d{3}|VRB)(\d{2,3})(G\d{2,3})?KT$'
    match = re.match(patron, cadena)
    
    if match:
        dir_raw = match.group(1)
        vel_raw = match.group(2)
        gus_raw = match.group(3)
        
        direccion = float(dir_raw) if dir_raw != 'VRB' else None
        velocidad = float(vel_raw)
        
        # CORRECCIÓN: Si no hay ráfaga (gus_raw es None), ponemos 0
        rafaga = float(gus_raw[1:]) if gus_raw else 0.0
        
        return pd.Series([direccion, velocidad, rafaga])
    
    return pd.Series([None, None, None])

In [100]:
# Convertir FECHA_HORA_REPORTE a datetime y establecer como índice
df = df_raw.copy()
df['FECHA_HORA_REPORTE'] = pd.to_datetime(df_raw['FECHA_HORA_REPORTE'])
df.set_index('FECHA_HORA_REPORTE', inplace=True)

# Eliminación de variables no necesarias, solo me quedo con la columna de viento
df = df[['viento']]

# Eliminar valores nulos en la columna 'viento' antes de aplicar el parseo
df = df[df['viento'].notna()]

# Modificar el valor KKT por KT en la columna de viento
df['viento'] = df['viento'].str.replace('KKT', 'KT', regex=False)

# Aplicación Regex para parsear la columna de viento con formato flexible
df[['direccion', 'intensidad_kt', 'rafaga_kt']] = df['viento'].apply(parse_viento_flexible)

# Toma el último grado válido (ej. 030) y lo pone donde estaba el NaN de 'VRB'
df['direccion'] = df['direccion'].ffill()

# Eliminacion de errores de digitación en la columna de viento, donde la longitud es mayor a 9 caracteres y la intensidad es NaN despues del parseo
df_2 = df[df['viento'].str.len() > 9]
df_2 = df_2[df_2["intensidad_kt"].isna()]
df = df.drop(df_2.index)

In [101]:
df.head()

,viento,direccion,intensidad_kt,rafaga_kt
FECHA_HORA_REPORTE,,,,
2006-07-20 00:00:00,03009KT,30.0,9.0,0.0
2006-07-20 01:00:00,25008KT,250.0,8.0,0.0
2006-07-20 05:00:00,VRB02KT,250.0,2.0,0.0
2006-07-20 06:00:00,29006KT,290.0,6.0,0.0
2006-07-20 07:00:00,32006KT,320.0,6.0,0.0


In [102]:
# Filtrar el dataset donde la columna "viento" su longitud es mayor a 8 y menor a 9
df_1 = df[df['viento'].str.len() > 8]
df_1 = df_1[df_1['viento'].str.len() < 13]
df_1

,viento,direccion,intensidad_kt,rafaga_kt
FECHA_HORA_REPORTE,,,,
2006-09-05 22:00:00,19007G18KT,190.0,7.0,18.0
2008-01-03 20:00:00,04016G30KT,40.0,16.0,30.0
2008-01-03 21:00:00,04015G25KT,40.0,15.0,25.0
2008-01-03 22:00:00,03014G25KT,30.0,14.0,25.0
2009-07-24 16:00:00,16016G30KT,160.0,16.0,30.0
...,...,...,...,...
2026-03-13 22:00:00,27010G21KT,270.0,10.0,21.0
2026-03-21 19:00:00,31012G23KT,310.0,12.0,23.0
2026-03-22 20:00:00,27009G19KT,270.0,9.0,19.0
